<a href="https://colab.research.google.com/github/seoha101127-cyber/kfa-ai-agent/blob/main/setup_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔧 환경점검 노트북

스터디에 필요한 세팅이 잘 됐는지 **한 번에** 확인하는 노트북이에요.

**이 노트북이 확인하는 것**
1. 필요한 라이브러리 설치 (라이브러리 = 남이 만들어 둔 코드 묶음)
2. API 키 등록 여부 (DART, Gemini)
3. 주가 받아오기 (삼성전자)
4. DART 공시 데이터 받아오기
5. Gemini(AI) 불러보기

**사용법**
1. 위쪽 메뉴에서 **런타임 → 모두 실행** 을 눌러요.
2. 중간에 "보안 비밀에 액세스하도록 허용할까요?" 같은 창이 뜨면 **허용**을 눌러요.
3. 다 끝나면 **맨 아래 "결과 요약"** 표를 확인해요.

> 💡 **1단계에서는 키가 없어서 ⏭(아직 안 함)가 나오는 게 정상이에요.**
> 키는 2회차 전까지 등록하면 되고, 등록한 뒤 이 노트북을 다시 돌리면 ✅로 바뀌어요.

In [ ]:
# ⚙️ 설정값 — 바꿀 일이 있으면 여기만 고치면 돼요
GEMINI_MODEL = "gemini-3.8-flash"   # 무료로 쓸 수 있는 Flash 계열 모델
GEMINI_FALLBACK_MODEL = "gemini-3.5-flash-lite"   # 서버가 붐빌 때 한 번 더 시도할 예비 모델 (Flash-Lite 계열)
SAMSUNG_STOCK_CODE = "005930"       # 삼성전자 종목코드 (주가 조회용)
SAMSUNG_CORP_CODE = "00126380"      # 삼성전자 DART 고유번호 (공시 조회용)

# 각 항목의 결과를 모아 두는 곳이에요. 맨 아래 요약에서 써요.
results = {}
print("✅ 설정 완료")

In [ ]:
# 📦 라이브러리 설치 (1~2분 걸릴 수 있어요)
import subprocess, sys

packages = ["google-genai", "finance-datareader", "pykrx", "python-dotenv"]
print("라이브러리를 설치하고 있어요. 잠시만 기다려 주세요...")

try:
    # 설치 중에 나오는 긴 글자들은 숨기고, 결과만 보여줘요
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        capture_output=True, text=True,
    )
    if proc.returncode == 0:
        results["라이브러리 설치"] = ("✅", "설치 완료")
        print("✅ 설치 완료")
    else:
        results["라이브러리 설치"] = ("❌", "설치 중 문제가 생겼어요")
        print("❌ 설치 중 문제가 생겼어요.")
        print("   '런타임 → 런타임 다시 시작'을 누른 뒤, 다시 '런타임 → 모두 실행'을 눌러 주세요.")
        print("   (참고용 내용)", proc.stderr.strip()[-300:])
except Exception as e:
    results["라이브러리 설치"] = ("❌", "설치 중 문제가 생겼어요")
    print("❌ 설치 중 문제가 생겼어요. '런타임 → 런타임 다시 시작' 후 다시 실행해 주세요.")
    print(f"   (참고용 내용) {type(e).__name__}")

In [ ]:
# 🔑 API 키 불러오기
# Colab이면 Secrets(🔑 보안 비밀)에서, 내 컴퓨터면 .env 파일에서 키를 읽어요.
# ⚠️ 키 값은 절대 화면에 출력하지 않아요.
import os

NO_KEY_MSG = "⏭ 아직 키 없음 (2회차 전까지 등록하면 돼요)"


def get_key(name):
    # 돌려주는 값: (키 값 또는 None, 상태 기호, 안내 문구)
    try:
        from google.colab import userdata   # Colab에서만 있는 기능
    except ImportError:
        userdata = None

    if userdata is not None:
        # Colab일 때: Secrets에서 읽기
        try:
            value = userdata.get(name)
        except userdata.SecretNotFoundError:
            return None, "⏭", NO_KEY_MSG
        except userdata.NotebookAccessError:
            return None, "⏭", "⏭ 등록은 됐는데, 왼쪽 🔑 아이콘에서 '노트북 액세스'를 켜 주세요"
    else:
        # 내 컴퓨터일 때: .env 파일에서 읽기
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            pass
        value = os.getenv(name)

    if not value:
        return None, "⏭", NO_KEY_MSG
    return value, "✅", "등록됨 ✅"


DART_API_KEY, GEMINI_API_KEY = None, None
for key_name, label in [("DART_API_KEY", "DART 키"), ("GEMINI_API_KEY", "Gemini 키")]:
    try:
        value, status, msg = get_key(key_name)
    except Exception as e:
        value, status, msg = None, "❌", "키를 읽는 중 문제가 생겼어요"
    if key_name == "DART_API_KEY":
        DART_API_KEY = value
    else:
        GEMINI_API_KEY = value
    results[label] = (status, msg.replace(status, "").strip())   # 요약 표에서는 기호를 빼고 보여줘요
    print(f"{label:10s}: {msg}")

In [ ]:
# 📈 주가 확인: 삼성전자 최근 5일
from datetime import datetime, timedelta

try:
    import FinanceDataReader as fdr
    # 연휴가 있어도 5일치가 나오도록 넉넉하게 한 달 전부터 받아요
    start = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")
    df = fdr.DataReader(SAMSUNG_STOCK_CODE, start)
    if df.empty:
        raise ValueError("받아온 주가가 비어 있어요")
    print("✅ 삼성전자 최근 5일 주가예요. (Close = 종가, 그날 마지막 가격)")
    display(df.tail(5))
    results["주가 조회"] = ("✅", "삼성전자 주가 받아오기 성공")
except Exception as e:
    results["주가 조회"] = ("❌", "주가를 받아오지 못했어요")
    print("❌ 주가를 받아오지 못했어요.")
    print("   인터넷 연결을 확인하고, 잠시 뒤 이 셀만 다시 실행해 주세요. (셀 클릭 → Shift + Enter)")
    print(f"   (참고용 내용) {type(e).__name__}: {e}")

In [ ]:
# 🏢 DART 확인: 삼성전자 기업개황 1회 조회
if DART_API_KEY is None:
    results["DART 조회"] = ("⏭", "키가 없어서 건너뛰었어요")
    print("⏭ DART 키가 없어서 건너뛰었어요. 키를 등록한 뒤 다시 실행하면 돼요.")
else:
    try:
        import requests
        resp = requests.get(
            "https://opendart.fss.or.kr/api/company.json",
            params={"crtfc_key": DART_API_KEY, "corp_code": SAMSUNG_CORP_CODE},
            timeout=15,
        )
        data = resp.json()
        if data.get("status") == "000":   # "000" = 정상
            results["DART 조회"] = ("✅", f"회사명: {data['corp_name']}")
            print(f"✅ DART 조회 성공! 회사명: {data['corp_name']}")
        else:
            dart_msg = data.get("message", "알 수 없는 응답")
            results["DART 조회"] = ("❌", f"DART가 요청을 거절했어요 ({dart_msg})")
            print(f"❌ DART가 요청을 거절했어요. (DART 안내: {dart_msg})")
            print("   🔑 Secrets에 키를 빠짐없이 정확히 복사해 넣었는지 확인해 주세요.")
    except Exception as e:
        results["DART 조회"] = ("❌", "DART에 연결하지 못했어요")
        print("❌ DART에 연결하지 못했어요. 잠시 뒤 이 셀만 다시 실행해 주세요.")
        # 에러 내용에 키가 섞여 나올 수 있어서, 에러 종류만 보여줘요
        print(f"   (참고용 내용) {type(e).__name__}")

In [ ]:
# 🤖 Gemini 확인: 1회 호출 (서버가 붐비면 자동으로 기다렸다가 다시 시도해요)
import time

if GEMINI_API_KEY is None:
    results["Gemini 호출"] = ("⏭", "키가 없어서 건너뛰었어요")
    print("⏭ Gemini 키가 없어서 건너뛰었어요. 키를 등록한 뒤 다시 실행하면 돼요.")
else:
    import logging
    logging.getLogger("google_genai").setLevel(logging.ERROR)   # 빨간 참고 메시지 숨기기
    from google import genai
    client = genai.Client(api_key=GEMINI_API_KEY)

    RETRY_CODES = (429, 500, 503)   # 기다리면 나아질 수 있는 에러 번호
    WAITS = [5, 10, 20]             # 다시 시도하기 전에 기다리는 시간(초). 점점 길어져요

    def ask_gemini(model):
        return client.models.generate_content(
            model=model,
            contents="안녕하세요에 한 문장으로 답해줘",
        )

    def ask_with_retry(model):
        # 붐비는 에러(429·500·503)면 기다렸다가 최대 3번 다시 시도해요
        for attempt in range(len(WAITS) + 1):
            try:
                return ask_gemini(model)
            except Exception as e:
                if getattr(e, "code", None) in RETRY_CODES and attempt < len(WAITS):
                    print(f"⏳ 잠깐 붐비는 것 같아요. {WAITS[attempt]}초 기다렸다가 다시 시도해요 ({attempt + 1}/{len(WAITS)})")
                    time.sleep(WAITS[attempt])
                else:
                    raise

    try:
        used_model = GEMINI_MODEL
        try:
            response = ask_with_retry(GEMINI_MODEL)
        except Exception as e:
            if getattr(e, "code", None) not in RETRY_CODES:
                raise
            # 3번 다시 시도해도 안 되면 예비 모델로 한 번 더 해봐요
            print(f"⏳ 계속 안 돼서 예비 모델({GEMINI_FALLBACK_MODEL})로 한 번 더 시도해요")
            used_model = GEMINI_FALLBACK_MODEL
            response = ask_gemini(GEMINI_FALLBACK_MODEL)
        answer = (response.text or "").strip()
        if used_model == GEMINI_MODEL:
            results["Gemini 호출"] = ("✅", "Gemini 답변 받기 성공")
        else:
            results["Gemini 호출"] = ("✅", f"예비 모델({used_model})로 답변 받기 성공")
        print(f"✅ Gemini 호출 성공! ({used_model})")
        print("   답변:", answer)
    except Exception as e:
        code_num = getattr(e, "code", None)   # 에러 번호
        if code_num == 503:
            results["Gemini 호출"] = ("❌", "서버 붐빔 (503)")
            print("❌ 구글 서버가 잠깐 붐비는 거예요. 내 설정 문제가 아니니 잠시 뒤 이 셀만 다시 실행해 주세요.")
        elif code_num == 500:
            results["Gemini 호출"] = ("❌", "서버 오류 (500)")
            print("❌ 구글 서버에 일시적인 문제가 있어요. 내 설정 문제가 아니니 잠시 뒤 이 셀만 다시 실행해 주세요.")
        elif code_num == 429:
            results["Gemini 호출"] = ("❌", "요청 제한 (429)")
            print("❌ 무료 사용량의 분당 요청 제한에 걸렸어요.")
            print("   1분 정도 기다렸다가 이 셀만 다시 실행해 주세요.")
        elif code_num in (400, 401, 403):
            results["Gemini 호출"] = ("❌", f"키가 올바르지 않은 것 같아요 ({code_num})")
            print("❌ Gemini가 키를 받아 주지 않았어요.")
            print("   🔑 Secrets에 키를 빠짐없이 정확히 복사해 넣었는지 확인해 주세요.")
        elif code_num == 404:
            results["Gemini 호출"] = ("❌", "모델 이름을 찾을 수 없어요 (404)")
            print(f"❌ 모델을 찾을 수 없어요. 스터디장에게 알려 주세요.")
        else:
            label = f"Gemini를 부르지 못했어요 ({code_num})" if code_num else "Gemini를 부르지 못했어요"
            results["Gemini 호출"] = ("❌", label)
            print("❌ Gemini를 부르지 못했어요. 잠시 뒤 이 셀만 다시 실행해 주세요.")
        print(f"   (참고용 내용) {type(e).__name__} {code_num or ''}")

In [ ]:
# 📋 결과 요약
import pandas as pd

ITEMS = ["라이브러리 설치", "DART 키", "Gemini 키", "주가 조회", "DART 조회", "Gemini 호출"]
GUIDE = {
    "라이브러리 설치": "런타임 다시 시작 → 모두 실행 (SETUP.md 12번)",
    "DART 키": "SETUP.md 8번(발급), 9번(Secrets 등록)",
    "Gemini 키": "SETUP.md 8번(발급), 9번(Secrets 등록)",
    "주가 조회": "인터넷·런타임 확인 (SETUP.md 12번)",
    "DART 조회": "SETUP.md 9번 (키 값을 정확히 넣었는지)",
    "Gemini 호출": "SETUP.md 9번 (키 값 확인) / 서버 붐빔(503)·요청 제한(429)이면 잠시 뒤 이 셀만 다시",
}

rows = []
for item in ITEMS:
    status, msg = results.get(item, ("❌", "이 항목이 실행되지 않았어요"))
    rows.append({
        "항목": item,
        "결과": status,
        "내용": msg,
        "❌일 때 볼 곳": GUIDE[item] if status == "❌" else "",
    })

display(pd.DataFrame(rows).set_index("항목"))

statuses = [r["결과"] for r in rows]
if "❌" in statuses:
    print("❌ 표시된 항목이 있어요. 오른쪽 '볼 곳'을 참고해서 고친 뒤, 다시 '런타임 → 모두 실행'을 눌러 주세요.")
elif "⏭" in statuses:
    print("👍 지금까지는 문제없어요! ⏭ 항목은 키를 등록한 뒤 다시 돌리면 ✅로 바뀌어요.")
else:
    print("🎉 모든 세팅이 끝났어요!")

## 📤 마지막: 내 레포에 올리기

결과를 확인했으면 이 노트북을 내 레포에 올려 주세요.

1. 위쪽 메뉴에서 **파일 → GitHub에 사본 저장** 을 눌러요.
2. **저장소**: 내 레포 `사용자이름/kfa-ai-agent` 를 골라요. (옆의 `main`은 그대로)
3. **파일 경로**: `setup_check.ipynb` 로 적어요.
   - 앞에 `사본_` 이나 `Copy of` 가 붙어 있으면 지워 주세요.
4. **커밋 메시지**: `환경점검` 처럼 한 줄 적고 **확인** 을 눌러요.

> 키를 나중에 등록했다면, 다시 "모두 실행"한 뒤 **같은 이름으로 또 올리면** 새 결과로 바뀌어요.
>
> 걱정 마세요. 이 노트북은 키 값을 화면에 출력하지 않아서, 올려도 키가 공개되지 않아요.